# Feature Engineering for Stress Detection

**CalmSense Project - Week 3**

This notebook demonstrates the comprehensive feature extraction pipeline for multimodal stress detection using the WESAD dataset. We extract 60+ physiological features from:

- **HRV (Heart Rate Variability)**: Time-domain, frequency-domain, and nonlinear features
- **EDA (Electrodermal Activity)**: Tonic (SCL) and phasic (SCR) components
- **Temperature**: Skin temperature statistics
- **Respiration**: Breathing rate and patterns
- **Accelerometer**: Movement and activity metrics

Additionally, we explore signal-to-image encoding techniques (GASF, GADF, MTF) for CNN-based classification.

---

## Table of Contents

1. [Setup and Imports](#1.-Setup-and-Imports)
2. [HRV Time-Domain Features](#2.-HRV-Time-Domain-Features)
3. [HRV Frequency-Domain Features](#3.-HRV-Frequency-Domain-Features)
4. [HRV Nonlinear Features](#4.-HRV-Nonlinear-Features)
5. [EDA Features](#5.-EDA-Features)
6. [Signal Image Encoding](#6.-Signal-Image-Encoding)
7. [Feature Correlation Analysis](#7.-Feature-Correlation-Analysis)
8. [Feature Distributions by Stress Condition](#8.-Feature-Distributions-by-Stress-Condition)
9. [Feature Summary Table](#9.-Feature-Summary-Table)

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal as scipy_signal

# CalmSense feature extractors
from src.features import (
    HRVTimeDomainExtractor,
    HRVFrequencyDomainExtractor,
    HRVNonlinearExtractor,
    EDAFeatureExtractor,
    TemperatureFeatureExtractor,
    RespirationFeatureExtractor,
    AccelerometerFeatureExtractor,
    SignalImageEncoder,
    FeatureExtractionPipeline
)

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Color palette for stress conditions
COLORS = {
    'baseline': '#2ecc71',
    'stress': '#e74c3c',
    'amusement': '#3498db'
}

print("CalmSense Feature Engineering Notebook")
print("=" * 40)

### Generate Synthetic Physiological Data

For demonstration, we generate synthetic data that mimics physiological signals during baseline and stress conditions.

In [ ]:
def generate_synthetic_rr_intervals(condition='baseline', n_beats=300, seed=42):
    """
    Generate synthetic RR intervals mimicking different stress conditions.
    
    Baseline: Higher HRV, lower HR (~70 BPM, SDNN ~50ms)
    Stress: Lower HRV, higher HR (~90 BPM, SDNN ~30ms)
    """
    np.random.seed(seed)
    
    if condition == 'baseline':
        mean_rr = 857  # ~70 BPM
        std_rr = 50
        # Add respiratory sinus arrhythmia (RSA)
        rsa_amplitude = 30
        rsa_freq = 0.25  # Hz (15 breaths/min)
    else:  # stress
        mean_rr = 667  # ~90 BPM
        std_rr = 30
        rsa_amplitude = 15  # Reduced RSA under stress
        rsa_freq = 0.30  # Faster breathing
    
    # Base RR intervals
    rr = np.random.normal(mean_rr, std_rr, n_beats)
    
    # Add RSA modulation
    t = np.cumsum(rr) / 1000  # Time in seconds
    rsa = rsa_amplitude * np.sin(2 * np.pi * rsa_freq * t)
    rr = rr + rsa
    
    # Ensure physiological range (300-1500 ms)
    rr = np.clip(rr, 300, 1500)
    
    return rr

def generate_synthetic_eda(condition='baseline', duration_sec=60, fs=4, seed=42):
    """
    Generate synthetic EDA signal with tonic and phasic components.
    
    Stress: Higher SCL, more frequent SCRs
    """
    np.random.seed(seed)
    n_samples = duration_sec * fs
    t = np.linspace(0, duration_sec, n_samples)
    
    if condition == 'baseline':
        scl_base = 2.0  # Lower tonic level
        scl_drift = 0.3
        n_scrs = 3
        scr_amplitude = 0.5
    else:  # stress
        scl_base = 5.0  # Higher tonic level
        scl_drift = 0.8
        n_scrs = 8  # More frequent SCRs
        scr_amplitude = 1.2
    
    # Tonic component (slow drift)
    tonic = scl_base + scl_drift * np.sin(2 * np.pi * 0.01 * t)
    tonic += 0.1 * np.random.randn(n_samples)  # Small noise
    
    # Phasic component (SCRs)
    phasic = np.zeros(n_samples)
    scr_times = np.random.choice(range(fs * 5, n_samples - fs * 5), n_scrs, replace=False)
    
    for scr_time in scr_times:
        # SCR waveform: fast rise, slow decay
        rise_samples = int(1.5 * fs)
        decay_samples = int(4 * fs)
        amplitude = scr_amplitude * (0.5 + np.random.rand())
        
        # Rise phase
        rise = np.linspace(0, amplitude, rise_samples)
        # Decay phase (exponential)
        decay = amplitude * np.exp(-np.linspace(0, 3, decay_samples))
        
        scr_wave = np.concatenate([rise, decay])
        end_idx = min(scr_time + len(scr_wave), n_samples)
        phasic[scr_time:end_idx] += scr_wave[:end_idx - scr_time]
    
    eda = tonic + phasic
    return eda, tonic, phasic

# Generate data for both conditions
rr_baseline = generate_synthetic_rr_intervals('baseline')
rr_stress = generate_synthetic_rr_intervals('stress')

eda_baseline, tonic_baseline, phasic_baseline = generate_synthetic_eda('baseline')
eda_stress, tonic_stress, phasic_stress = generate_synthetic_eda('stress')

print(f"Generated {len(rr_baseline)} baseline RR intervals")
print(f"Generated {len(rr_stress)} stress RR intervals")
print(f"Generated {len(eda_baseline)} EDA samples per condition")

---

## 2. HRV Time-Domain Features

Time-domain HRV features quantify the amount of variability in inter-beat intervals (RR intervals). These are among the most commonly used HRV metrics.

### Mathematical Formulas

| Feature | Formula | Description |
|---------|---------|-------------|
| **MeanNN** | $\bar{RR} = \frac{1}{N}\sum_{i=1}^{N}RR_i$ | Mean of RR intervals |
| **SDNN** | $\sqrt{\frac{1}{N-1}\sum_{i=1}^{N}(RR_i - \bar{RR})^2}$ | Standard deviation of NN intervals |
| **RMSSD** | $\sqrt{\frac{1}{N-1}\sum_{i=1}^{N-1}(RR_{i+1} - RR_i)^2}$ | Root mean square of successive differences |
| **pNN50** | $\frac{\text{count}(|\Delta RR| > 50\text{ms})}{N-1} \times 100$ | Percentage of successive differences > 50ms |
| **HRVTI** | $\frac{N}{\text{height of histogram mode}}$ | HRV Triangular Index |

*Reference: Task Force of ESC and NASPE (1996). Circulation, 93(5), 1043-1065.*

In [ ]:
# Initialize HRV Time-Domain Extractor
hrv_time_extractor = HRVTimeDomainExtractor()

# Extract features for both conditions
hrv_time_baseline = hrv_time_extractor.extract_all(rr_baseline)
hrv_time_stress = hrv_time_extractor.extract_all(rr_stress)

# Create comparison table
hrv_time_df = pd.DataFrame({
    'Feature': list(hrv_time_baseline.keys()),
    'Baseline': [f"{v:.2f}" for v in hrv_time_baseline.values()],
    'Stress': [f"{v:.2f}" for v in hrv_time_stress.values()],
    'Unit': ['ms', 'ms', 'ms', '%', '%', '', '', 'ms', '', 'ms', '', 'ms']
})

# Add descriptions
descriptions = hrv_time_extractor.get_feature_descriptions()
hrv_time_df['Description'] = hrv_time_df['Feature'].map(descriptions)

print("HRV Time-Domain Features Comparison")
print("=" * 60)
display(hrv_time_df)

In [ ]:
# Visualize key HRV time-domain features
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Plot 1: RR interval distribution
ax1 = axes[0]
ax1.hist(rr_baseline, bins=30, alpha=0.7, label='Baseline', color=COLORS['baseline'], edgecolor='white')
ax1.hist(rr_stress, bins=30, alpha=0.7, label='Stress', color=COLORS['stress'], edgecolor='white')
ax1.axvline(np.mean(rr_baseline), color=COLORS['baseline'], linestyle='--', linewidth=2)
ax1.axvline(np.mean(rr_stress), color=COLORS['stress'], linestyle='--', linewidth=2)
ax1.set_xlabel('RR Interval (ms)')
ax1.set_ylabel('Count')
ax1.set_title('RR Interval Distribution')
ax1.legend()

# Plot 2: SDNN comparison (bar chart)
ax2 = axes[1]
features_to_compare = ['SDNN', 'RMSSD', 'SDSD']
x = np.arange(len(features_to_compare))
width = 0.35

baseline_vals = [hrv_time_baseline[f] for f in features_to_compare]
stress_vals = [hrv_time_stress[f] for f in features_to_compare]

bars1 = ax2.bar(x - width/2, baseline_vals, width, label='Baseline', color=COLORS['baseline'])
bars2 = ax2.bar(x + width/2, stress_vals, width, label='Stress', color=COLORS['stress'])

ax2.set_xlabel('Feature')
ax2.set_ylabel('Value (ms)')
ax2.set_title('HRV Variability Metrics')
ax2.set_xticks(x)
ax2.set_xticklabels(features_to_compare)
ax2.legend()

# Add value labels on bars
for bar in bars1:
    ax2.annotate(f'{bar.get_height():.1f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)
for bar in bars2:
    ax2.annotate(f'{bar.get_height():.1f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

# Plot 3: pNN50 and pNN20
ax3 = axes[2]
pnn_features = ['pNN50', 'pNN20']
x = np.arange(len(pnn_features))

baseline_pnn = [hrv_time_baseline[f] for f in pnn_features]
stress_pnn = [hrv_time_stress[f] for f in pnn_features]

bars1 = ax3.bar(x - width/2, baseline_pnn, width, label='Baseline', color=COLORS['baseline'])
bars2 = ax3.bar(x + width/2, stress_pnn, width, label='Stress', color=COLORS['stress'])

ax3.set_xlabel('Feature')
ax3.set_ylabel('Percentage (%)')
ax3.set_title('Successive Difference Metrics')
ax3.set_xticks(x)
ax3.set_xticklabels(pnn_features)
ax3.legend()

plt.tight_layout()
plt.savefig('../figures/hrv_time_domain_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n Key Observations:")
print(f"  - SDNN decreases from {hrv_time_baseline['SDNN']:.1f}ms (baseline) to {hrv_time_stress['SDNN']:.1f}ms (stress)")
print(f"  - RMSSD decreases from {hrv_time_baseline['RMSSD']:.1f}ms to {hrv_time_stress['RMSSD']:.1f}ms")
print(f"  - Lower HRV during stress indicates reduced parasympathetic activity")

---

## 3. HRV Frequency-Domain Features

Frequency-domain analysis reveals the rhythmic components of HRV through Power Spectral Density (PSD) estimation.

### Standard Frequency Bands (Task Force 1996)

| Band | Frequency Range | Physiological Significance |
|------|-----------------|---------------------------|
| **VLF** | 0.0033 - 0.04 Hz | Thermoregulation, RAAS, slow mechanisms |
| **LF** | 0.04 - 0.15 Hz | Mixed sympathetic + parasympathetic |
| **HF** | 0.15 - 0.40 Hz | Parasympathetic (vagal) activity, RSA |

### Key Metrics

- **LF/HF Ratio**: Sympathovagal balance indicator (↑ with stress)
- **HF Power (nu)**: Normalized HF = HF / (LF + HF) × 100

In [ ]:
# Initialize HRV Frequency-Domain Extractor
hrv_freq_extractor = HRVFrequencyDomainExtractor()

# Extract features
hrv_freq_baseline = hrv_freq_extractor.extract_all(rr_baseline)
hrv_freq_stress = hrv_freq_extractor.extract_all(rr_stress)

# Display comparison
hrv_freq_df = pd.DataFrame({
    'Feature': list(hrv_freq_baseline.keys()),
    'Baseline': [f"{v:.4f}" if v < 1 else f"{v:.2f}" for v in hrv_freq_baseline.values()],
    'Stress': [f"{v:.4f}" if v < 1 else f"{v:.2f}" for v in hrv_freq_stress.values()]
})

print("HRV Frequency-Domain Features Comparison")
print("=" * 60)
display(hrv_freq_df)

In [ ]:
# Compute and visualize PSD for both conditions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

def compute_psd_for_plot(rr_intervals, fs_interp=4.0):
    """Compute PSD using Welch's method with interpolation."""
    # Create time vector
    t_rr = np.cumsum(rr_intervals) / 1000.0
    t_rr = t_rr - t_rr[0]
    
    # Interpolate to uniform sampling
    t_interp = np.arange(0, t_rr[-1], 1/fs_interp)
    rr_interp = np.interp(t_interp, t_rr, rr_intervals)
    
    # Remove mean
    rr_interp = rr_interp - np.mean(rr_interp)
    
    # Compute PSD
    freqs, psd = scipy_signal.welch(rr_interp, fs=fs_interp, nperseg=min(256, len(rr_interp)//2))
    
    return freqs, psd

freqs_bl, psd_bl = compute_psd_for_plot(rr_baseline)
freqs_st, psd_st = compute_psd_for_plot(rr_stress)

# Plot 1: Full PSD comparison
ax1 = axes[0]
ax1.semilogy(freqs_bl, psd_bl, label='Baseline', color=COLORS['baseline'], linewidth=2)
ax1.semilogy(freqs_st, psd_st, label='Stress', color=COLORS['stress'], linewidth=2)

# Add frequency band shading
ax1.axvspan(0.0033, 0.04, alpha=0.2, color='gray', label='VLF')
ax1.axvspan(0.04, 0.15, alpha=0.2, color='orange', label='LF')
ax1.axvspan(0.15, 0.40, alpha=0.2, color='cyan', label='HF')

ax1.set_xlim(0, 0.5)
ax1.set_xlabel('Frequency (Hz)')
ax1.set_ylabel('PSD (ms²/Hz)')
ax1.set_title('Power Spectral Density')
ax1.legend(loc='upper right')

# Plot 2: Band powers comparison
ax2 = axes[1]
bands = ['VLF_power', 'LF_power', 'HF_power']
band_labels = ['VLF', 'LF', 'HF']
x = np.arange(len(bands))
width = 0.35

baseline_powers = [hrv_freq_baseline[b] for b in bands]
stress_powers = [hrv_freq_stress[b] for b in bands]

ax2.bar(x - width/2, baseline_powers, width, label='Baseline', color=COLORS['baseline'])
ax2.bar(x + width/2, stress_powers, width, label='Stress', color=COLORS['stress'])

ax2.set_xlabel('Frequency Band')
ax2.set_ylabel('Power (ms²)')
ax2.set_title('Band Power Comparison')
ax2.set_xticks(x)
ax2.set_xticklabels(band_labels)
ax2.legend()

# Plot 3: LF/HF Ratio
ax3 = axes[2]
ratios = [hrv_freq_baseline['LF_HF_ratio'], hrv_freq_stress['LF_HF_ratio']]
conditions = ['Baseline', 'Stress']
colors = [COLORS['baseline'], COLORS['stress']]

bars = ax3.bar(conditions, ratios, color=colors, edgecolor='black', linewidth=1.5)
ax3.set_ylabel('LF/HF Ratio')
ax3.set_title('Sympathovagal Balance')

# Add value labels
for bar, val in zip(bars, ratios):
    ax3.annotate(f'{val:.2f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 xytext=(0, 5), textcoords='offset points', ha='center', fontsize=12, fontweight='bold')

# Add reference line for balanced state
ax3.axhline(y=1.0, color='gray', linestyle='--', label='Balanced (1.0)')
ax3.legend()

plt.tight_layout()
plt.savefig('../figures/hrv_frequency_domain_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n Key Observations:")
print(f"  - LF/HF ratio increases from {hrv_freq_baseline['LF_HF_ratio']:.2f} to {hrv_freq_stress['LF_HF_ratio']:.2f} under stress")
print(f"  - HF power (parasympathetic) decreases during stress")
print(f"  - Higher LF/HF indicates sympathetic dominance")

---

## 4. HRV Nonlinear Features

Nonlinear analysis captures the complexity and self-similarity of heart rate dynamics that linear methods cannot detect.

### Key Methods

1. **Poincaré Plot Analysis**
   - SD1: Short-term variability (perpendicular to line of identity)
   - SD2: Long-term variability (along line of identity)
   - CSI = SD2/SD1: Cardiac Sympathetic Index
   - CVI = log(SD1 × SD2): Cardiac Vagal Index

2. **Entropy Measures**
   - Sample Entropy (SampEn): Regularity/predictability
   - Approximate Entropy (ApEn): Complexity measure

3. **Detrended Fluctuation Analysis (DFA)**
   - α1 (4-16 beats): Short-term fractal scaling
   - α2 (16-64 beats): Long-term fractal scaling

In [ ]:
# Initialize HRV Nonlinear Extractor
hrv_nl_extractor = HRVNonlinearExtractor()

# Extract features
hrv_nl_baseline = hrv_nl_extractor.extract_all(rr_baseline)
hrv_nl_stress = hrv_nl_extractor.extract_all(rr_stress)

# Display comparison
hrv_nl_df = pd.DataFrame({
    'Feature': list(hrv_nl_baseline.keys()),
    'Baseline': [f"{v:.4f}" for v in hrv_nl_baseline.values()],
    'Stress': [f"{v:.4f}" for v in hrv_nl_stress.values()]
})

print("HRV Nonlinear Features Comparison")
print("=" * 60)
display(hrv_nl_df)

In [ ]:
# Visualize Poincaré plots and DFA
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Poincaré plot - Baseline
ax1 = axes[0]
rr_n = rr_baseline[:-1]
rr_n1 = rr_baseline[1:]
ax1.scatter(rr_n, rr_n1, alpha=0.5, color=COLORS['baseline'], s=20)

# Add line of identity
lim_min = min(rr_n.min(), rr_n1.min()) - 20
lim_max = max(rr_n.max(), rr_n1.max()) + 20
ax1.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', alpha=0.5, label='Line of identity')

# Add SD1/SD2 ellipse
from matplotlib.patches import Ellipse
mean_rr = np.mean(rr_baseline)
sd1 = hrv_nl_baseline['SD1']
sd2 = hrv_nl_baseline['SD2']
ellipse = Ellipse((mean_rr, mean_rr), width=2*sd2, height=2*sd1, angle=45,
                  fill=False, color='black', linewidth=2)
ax1.add_patch(ellipse)

ax1.set_xlabel('RR(n) [ms]')
ax1.set_ylabel('RR(n+1) [ms]')
ax1.set_title(f'Poincaré Plot - Baseline\nSD1={sd1:.1f}, SD2={sd2:.1f}')
ax1.set_xlim(lim_min, lim_max)
ax1.set_ylim(lim_min, lim_max)
ax1.set_aspect('equal')

# Plot 2: Poincaré plot - Stress
ax2 = axes[1]
rr_n = rr_stress[:-1]
rr_n1 = rr_stress[1:]
ax2.scatter(rr_n, rr_n1, alpha=0.5, color=COLORS['stress'], s=20)

# Add line of identity
lim_min = min(rr_n.min(), rr_n1.min()) - 20
lim_max = max(rr_n.max(), rr_n1.max()) + 20
ax2.plot([lim_min, lim_max], [lim_min, lim_max], 'k--', alpha=0.5)

# Add SD1/SD2 ellipse
mean_rr = np.mean(rr_stress)
sd1 = hrv_nl_stress['SD1']
sd2 = hrv_nl_stress['SD2']
ellipse = Ellipse((mean_rr, mean_rr), width=2*sd2, height=2*sd1, angle=45,
                  fill=False, color='black', linewidth=2)
ax2.add_patch(ellipse)

ax2.set_xlabel('RR(n) [ms]')
ax2.set_ylabel('RR(n+1) [ms]')
ax2.set_title(f'Poincaré Plot - Stress\nSD1={sd1:.1f}, SD2={sd2:.1f}')
ax2.set_xlim(lim_min, lim_max)
ax2.set_ylim(lim_min, lim_max)
ax2.set_aspect('equal')

# Plot 3: DFA comparison
ax3 = axes[2]
dfa_features = ['DFA_alpha1', 'DFA_alpha2']
x = np.arange(len(dfa_features))
width = 0.35

baseline_dfa = [hrv_nl_baseline[f] for f in dfa_features]
stress_dfa = [hrv_nl_stress[f] for f in dfa_features]

bars1 = ax3.bar(x - width/2, baseline_dfa, width, label='Baseline', color=COLORS['baseline'])
bars2 = ax3.bar(x + width/2, stress_dfa, width, label='Stress', color=COLORS['stress'])

ax3.axhline(y=1.0, color='gray', linestyle='--', label='α=1 (1/f noise)')
ax3.axhline(y=0.5, color='lightgray', linestyle=':', label='α=0.5 (white noise)')
ax3.axhline(y=1.5, color='lightgray', linestyle=':', label='α=1.5 (Brownian)')

ax3.set_xlabel('DFA Scale')
ax3.set_ylabel('Scaling Exponent (α)')
ax3.set_title('Detrended Fluctuation Analysis')
ax3.set_xticks(x)
ax3.set_xticklabels(['α1 (short-term)', 'α2 (long-term)'])
ax3.legend(loc='upper right', fontsize=8)
ax3.set_ylim(0, 2)

plt.tight_layout()
plt.savefig('../figures/hrv_nonlinear_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n Key Observations:")
print(f"  - Poincaré ellipse is smaller under stress (reduced variability)")
print(f"  - SD1/SD2 ratio changes reflect altered autonomic balance")
print(f"  - DFA α1 closer to 1.0 indicates healthy fractal dynamics")

---

## 5. EDA Features

Electrodermal Activity (EDA) reflects sympathetic nervous system activity through sweat gland innervation.

### Components

- **Tonic (SCL)**: Slow-varying baseline skin conductance level
- **Phasic (SCR)**: Rapid skin conductance responses to stimuli

### Stress Indicators

| Metric | Stress Effect |
|--------|---------------|
| SCL Mean | ↑ Increases |
| SCR Rate | ↑ More frequent |
| SCR Amplitude | ↑ Larger responses |

In [ ]:
# Initialize EDA Feature Extractor
eda_extractor = EDAFeatureExtractor(sampling_rate=4)

# Extract features
eda_features_baseline = eda_extractor.extract_all(
    {'tonic': tonic_baseline, 'phasic': phasic_baseline},
    scr_peaks=None,
    raw_eda=eda_baseline
)

eda_features_stress = eda_extractor.extract_all(
    {'tonic': tonic_stress, 'phasic': phasic_stress},
    scr_peaks=None,
    raw_eda=eda_stress
)

# Display comparison
eda_df = pd.DataFrame({
    'Feature': list(eda_features_baseline.keys()),
    'Baseline': [f"{v:.3f}" for v in eda_features_baseline.values()],
    'Stress': [f"{v:.3f}" for v in eda_features_stress.values()]
})

print("EDA Features Comparison")
print("=" * 60)
display(eda_df)

In [ ]:
# Visualize EDA signals and decomposition
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

fs = 4
t = np.arange(len(eda_baseline)) / fs

# Plot 1: Raw EDA - Baseline
ax1 = axes[0, 0]
ax1.plot(t, eda_baseline, color=COLORS['baseline'], linewidth=1.5, label='Raw EDA')
ax1.plot(t, tonic_baseline, color='darkgreen', linewidth=2, linestyle='--', label='Tonic (SCL)')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Skin Conductance (µS)')
ax1.set_title('EDA Signal - Baseline')
ax1.legend()
ax1.set_xlim(0, t[-1])

# Plot 2: Raw EDA - Stress
ax2 = axes[0, 1]
ax2.plot(t, eda_stress, color=COLORS['stress'], linewidth=1.5, label='Raw EDA')
ax2.plot(t, tonic_stress, color='darkred', linewidth=2, linestyle='--', label='Tonic (SCL)')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Skin Conductance (µS)')
ax2.set_title('EDA Signal - Stress')
ax2.legend()
ax2.set_xlim(0, t[-1])

# Plot 3: Phasic comparison
ax3 = axes[1, 0]
ax3.plot(t, phasic_baseline, color=COLORS['baseline'], linewidth=1.5, label='Baseline', alpha=0.8)
ax3.plot(t, phasic_stress, color=COLORS['stress'], linewidth=1.5, label='Stress', alpha=0.8)
ax3.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Phasic Component (µS)')
ax3.set_title('Phasic (SCR) Component Comparison')
ax3.legend()
ax3.set_xlim(0, t[-1])

# Plot 4: Key EDA metrics comparison
ax4 = axes[1, 1]
metrics = ['SCL_mean', 'EDA_mean', 'EDA_range']
metric_labels = ['SCL Mean', 'EDA Mean', 'EDA Range']
x = np.arange(len(metrics))
width = 0.35

baseline_vals = [eda_features_baseline[m] for m in metrics]
stress_vals = [eda_features_stress[m] for m in metrics]

bars1 = ax4.bar(x - width/2, baseline_vals, width, label='Baseline', color=COLORS['baseline'])
bars2 = ax4.bar(x + width/2, stress_vals, width, label='Stress', color=COLORS['stress'])

ax4.set_xlabel('Metric')
ax4.set_ylabel('Value (µS)')
ax4.set_title('EDA Metrics Comparison')
ax4.set_xticks(x)
ax4.set_xticklabels(metric_labels)
ax4.legend()

plt.tight_layout()
plt.savefig('../figures/eda_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n Key Observations:")
print(f"  - SCL (tonic) is higher under stress: {eda_features_stress['SCL_mean']:.2f} vs {eda_features_baseline['SCL_mean']:.2f} µS")
print(f"  - More frequent SCRs visible in phasic component during stress")
print(f"  - Larger SCR amplitudes indicate stronger sympathetic responses")

---

## 6. Signal Image Encoding

Signal-to-image encoding transforms 1D time series into 2D images for CNN-based classification.

### Methods

1. **GASF (Gramian Angular Summation Field)**
   - Encodes temporal correlation: $GASF[i,j] = \cos(\phi_i + \phi_j)$
   
2. **GADF (Gramian Angular Difference Field)**
   - Encodes temporal difference: $GADF[i,j] = \sin(\phi_i - \phi_j)$
   
3. **MTF (Markov Transition Field)**
   - Encodes transition probabilities between quantile bins

*Reference: Wang & Oates (2015). IJCAI.*

In [ ]:
# Initialize Signal Image Encoder
encoder = SignalImageEncoder(image_size=64)

# Use a segment of RR intervals for encoding
signal_segment = rr_baseline[:100]

# Encode using all methods
gasf = encoder.encode_gasf(signal_segment)
gadf = encoder.encode_gadf(signal_segment)
mtf = encoder.encode_mtf(signal_segment)
rgb = encoder.encode_rgb(signal_segment)
rp = encoder.encode_recurrence_plot(signal_segment)

print(f"GASF shape: {gasf.shape}")
print(f"GADF shape: {gadf.shape}")
print(f"MTF shape: {mtf.shape}")
print(f"RGB (stacked) shape: {rgb.shape}")

In [ ]:
# Visualize all encoding methods
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Original signal
ax0 = axes[0, 0]
ax0.plot(signal_segment, color='navy', linewidth=1.5)
ax0.set_xlabel('Beat Index')
ax0.set_ylabel('RR Interval (ms)')
ax0.set_title('Original RR Interval Signal')
ax0.grid(True, alpha=0.3)

# GASF
ax1 = axes[0, 1]
im1 = ax1.imshow(gasf, cmap='viridis', aspect='auto')
ax1.set_title('GASF (Gramian Angular Summation Field)')
ax1.set_xlabel('Time Index')
ax1.set_ylabel('Time Index')
plt.colorbar(im1, ax=ax1, fraction=0.046)

# GADF
ax2 = axes[0, 2]
im2 = ax2.imshow(gadf, cmap='RdBu_r', aspect='auto')
ax2.set_title('GADF (Gramian Angular Difference Field)')
ax2.set_xlabel('Time Index')
ax2.set_ylabel('Time Index')
plt.colorbar(im2, ax=ax2, fraction=0.046)

# MTF
ax3 = axes[1, 0]
im3 = ax3.imshow(mtf, cmap='hot', aspect='auto')
ax3.set_title('MTF (Markov Transition Field)')
ax3.set_xlabel('Time Index')
ax3.set_ylabel('Time Index')
plt.colorbar(im3, ax=ax3, fraction=0.046)

# Recurrence Plot
ax4 = axes[1, 1]
im4 = ax4.imshow(rp, cmap='binary', aspect='auto')
ax4.set_title('Recurrence Plot')
ax4.set_xlabel('Time Index')
ax4.set_ylabel('Time Index')
plt.colorbar(im4, ax=ax4, fraction=0.046)

# RGB Combined
ax5 = axes[1, 2]
ax5.imshow(rgb)
ax5.set_title('RGB Stack (R=GASF, G=GADF, B=MTF)')
ax5.set_xlabel('Time Index')
ax5.set_ylabel('Time Index')

plt.tight_layout()
plt.savefig('../figures/signal_image_encoding.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n Encoding Methods:")
print("  - GASF: Captures temporal correlation patterns")
print("  - GADF: Captures directional changes over time")
print("  - MTF: Captures state transition dynamics")
print("  - RGB: Combined representation for CNN input")

In [ ]:
# Compare encodings for baseline vs stress
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

# Baseline encodings
gasf_bl = encoder.encode_gasf(rr_baseline[:100])
gadf_bl = encoder.encode_gadf(rr_baseline[:100])
mtf_bl = encoder.encode_mtf(rr_baseline[:100])

# Stress encodings
gasf_st = encoder.encode_gasf(rr_stress[:100])
gadf_st = encoder.encode_gadf(rr_stress[:100])
mtf_st = encoder.encode_mtf(rr_stress[:100])

# Row 1: Baseline
axes[0, 0].imshow(gasf_bl, cmap='viridis')
axes[0, 0].set_title('Baseline - GASF')
axes[0, 1].imshow(gadf_bl, cmap='RdBu_r')
axes[0, 1].set_title('Baseline - GADF')
axes[0, 2].imshow(mtf_bl, cmap='hot')
axes[0, 2].set_title('Baseline - MTF')

# Row 2: Stress
axes[1, 0].imshow(gasf_st, cmap='viridis')
axes[1, 0].set_title('Stress - GASF')
axes[1, 1].imshow(gadf_st, cmap='RdBu_r')
axes[1, 1].set_title('Stress - GADF')
axes[1, 2].imshow(mtf_st, cmap='hot')
axes[1, 2].set_title('Stress - MTF')

for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Signal Image Encoding: Baseline vs Stress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/encoding_baseline_vs_stress.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisual differences between conditions are suitable for CNN classification.")

---

## 7. Feature Correlation Analysis

Understanding feature correlations helps identify redundant features and inform feature selection strategies.

In [ ]:
# Initialize the complete feature extraction pipeline
pipeline = FeatureExtractionPipeline()

# Get feature names and count
feature_names = pipeline.get_feature_names()
print(f"Total features in pipeline: {len(feature_names)}")
print(f"\nFeature groups:")
for group, features in pipeline.get_feature_groups().items():
    print(f"  {group}: {len(features)} features")

In [ ]:
# Generate multiple samples for correlation analysis
np.random.seed(42)
n_samples = 50

all_features = []

for i in range(n_samples):
    # Alternate between baseline and stress
    condition = 'baseline' if i % 2 == 0 else 'stress'
    
    # Generate synthetic data
    rr = generate_synthetic_rr_intervals(condition, seed=i)
    eda, tonic, phasic = generate_synthetic_eda(condition, seed=i)
    
    # Create window data
    window_data = {
        'rr_intervals': rr,
        'eda_tonic': tonic,
        'eda_phasic': phasic,
        'eda_raw': eda,
        'temperature': 33 + np.random.randn(100) * 0.5,
        'respiration': np.sin(np.linspace(0, 10*np.pi, 1000)) + 0.1*np.random.randn(1000),
        'accelerometer': {'magnitude': 1 + 0.2*np.random.randn(100)},
        'label': 0 if condition == 'baseline' else 1
    }
    
    # Extract features
    features = pipeline.extract_window_features(window_data)
    features['condition'] = condition
    all_features.append(features)

# Create DataFrame
features_df = pd.DataFrame(all_features)
print(f"Generated {len(features_df)} feature samples")
print(f"Features per sample: {len(features_df.columns) - 1}")

In [ ]:
# Compute correlation matrix for numeric features only
numeric_cols = features_df.select_dtypes(include=[np.number]).columns
corr_matrix = features_df[numeric_cols].corr()

# Plot correlation heatmap
fig, ax = plt.subplots(figsize=(16, 14))

# Create mask for upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Custom colormap
cmap = sns.diverging_palette(250, 10, as_cmap=True)

# Plot heatmap
sns.heatmap(corr_matrix, mask=mask, cmap=cmap, center=0,
            square=True, linewidths=0.5, annot=False,
            cbar_kws={"shrink": 0.5, "label": "Correlation"},
            xticklabels=True, yticklabels=True)

plt.title('Feature Correlation Matrix (60+ Features)', fontsize=14, fontweight='bold')
plt.xticks(rotation=90, fontsize=6)
plt.yticks(fontsize=6)

plt.tight_layout()
plt.savefig('../figures/feature_correlation_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

# Find highly correlated feature pairs
high_corr_threshold = 0.8
high_corr_pairs = []

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > high_corr_threshold:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

print(f"\nHighly correlated feature pairs (|r| > {high_corr_threshold}):")
for f1, f2, r in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True)[:10]:
    print(f"  {f1} <-> {f2}: r={r:.3f}")

---

## 8. Feature Distributions by Stress Condition

Visualizing feature distributions helps understand discriminative power for stress classification.

In [ ]:
# Select key features for violin plots
key_features = [
    'HRV_SDNN', 'HRV_RMSSD', 'HRV_pNN50', 'HRV_LF_HF_ratio',
    'HRV_SD1', 'HRV_SampEn', 'EDA_SCL_mean', 'EDA_EDA_range'
]

# Filter to available features
available_features = [f for f in key_features if f in features_df.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, feature in enumerate(available_features):
    ax = axes[idx]
    
    # Create violin plot
    parts = ax.violinplot(
        [features_df[features_df['condition'] == 'baseline'][feature].dropna(),
         features_df[features_df['condition'] == 'stress'][feature].dropna()],
        positions=[0, 1],
        showmeans=True,
        showmedians=True
    )
    
    # Color the violins
    for i, pc in enumerate(parts['bodies']):
        color = COLORS['baseline'] if i == 0 else COLORS['stress']
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Baseline', 'Stress'])
    ax.set_title(feature.replace('HRV_', '').replace('EDA_', ''), fontsize=11)
    ax.grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(len(available_features), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Feature Distributions: Baseline vs Stress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/feature_distributions_violin.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical comparison: Effect sizes and significance
from scipy import stats

print("Feature Discriminative Power (Baseline vs Stress)")
print("=" * 70)
print(f"{'Feature':<25} {'Baseline Mean':>12} {'Stress Mean':>12} {'Cohen\'s d':>10} {'p-value':>10}")
print("-" * 70)

for feature in available_features:
    baseline_vals = features_df[features_df['condition'] == 'baseline'][feature].dropna()
    stress_vals = features_df[features_df['condition'] == 'stress'][feature].dropna()
    
    if len(baseline_vals) > 1 and len(stress_vals) > 1:
        # T-test
        t_stat, p_val = stats.ttest_ind(baseline_vals, stress_vals)
        
        # Cohen's d effect size
        pooled_std = np.sqrt(((len(baseline_vals)-1)*baseline_vals.std()**2 + 
                              (len(stress_vals)-1)*stress_vals.std()**2) / 
                             (len(baseline_vals) + len(stress_vals) - 2))
        cohens_d = (stress_vals.mean() - baseline_vals.mean()) / pooled_std if pooled_std > 0 else 0
        
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else ""
        
        print(f"{feature:<25} {baseline_vals.mean():>12.3f} {stress_vals.mean():>12.3f} {cohens_d:>10.2f} {p_val:>9.4f}{sig}")

print("\nSignificance: *** p<0.001, ** p<0.01, * p<0.05")
print("Cohen's d interpretation: |d| < 0.2 small, 0.2-0.8 medium, > 0.8 large")

---

## 9. Feature Summary Table

Complete reference of all 60+ features extracted by the CalmSense pipeline.

In [ ]:
# Get all feature descriptions
all_descriptions = pipeline.get_feature_descriptions()

# Create summary table
summary_data = []

for feature, description in all_descriptions.items():
    # Determine group
    if feature.startswith('HRV_'):
        base = feature[4:]
        if base in ['MeanNN', 'SDNN', 'RMSSD', 'SDSD', 'pNN50', 'pNN20', 'CVNN', 
                    'CVSD', 'MadNN', 'MCVNN', 'IQRNN', 'HRVTI']:
            group = 'HRV Time-Domain'
        elif base in ['VLF_power', 'LF_power', 'HF_power', 'LF_HF_ratio', 
                      'LF_nu', 'HF_nu', 'Total_power', 'VLF_peak']:
            group = 'HRV Frequency-Domain'
        else:
            group = 'HRV Nonlinear'
    elif feature.startswith('EDA_'):
        group = 'EDA'
    elif feature.startswith('TEMP_'):
        group = 'Temperature'
    elif feature.startswith('RESP_'):
        group = 'Respiration'
    elif feature.startswith('ACC_'):
        group = 'Accelerometer'
    else:
        group = 'Other'
    
    summary_data.append({
        'Feature': feature,
        'Group': group,
        'Description': description
    })

summary_df = pd.DataFrame(summary_data)

# Display by group
print("COMPLETE FEATURE REFERENCE")
print("=" * 80)

for group in ['HRV Time-Domain', 'HRV Frequency-Domain', 'HRV Nonlinear', 
              'EDA', 'Temperature', 'Respiration', 'Accelerometer']:
    group_features = summary_df[summary_df['Group'] == group]
    print(f"\n{group} ({len(group_features)} features)")
    print("-" * 60)
    for _, row in group_features.iterrows():
        print(f"  {row['Feature']:<30} {row['Description']}")

print(f"\n{'='*80}")
print(f"TOTAL FEATURES: {len(summary_df)}")

In [ ]:
# Save feature summary to CSV for documentation
summary_df.to_csv('../data/feature_summary.csv', index=False)
print("Feature summary saved to ../data/feature_summary.csv")

# Display styled table
display(summary_df.style.set_properties(**{
    'text-align': 'left'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left')]}
]))

---

## Summary

This notebook demonstrated the comprehensive feature extraction pipeline for the CalmSense stress detection project:

### Key Achievements

1. **60+ Physiological Features** extracted from multimodal signals
2. **Signal Image Encoding** (GASF, GADF, MTF) for CNN-based classification
3. **Feature Analysis** showing clear differentiation between stress conditions

### Feature Groups

| Group | Count | Key Stress Indicators |
|-------|-------|----------------------|
| HRV Time-Domain | 12 | SDNN↓, RMSSD↓, pNN50↓ |
| HRV Frequency-Domain | 8 | LF/HF↑, HF↓ |
| HRV Nonlinear | 10 | SD1↓, SampEn varies |
| EDA | 15 | SCL↑, SCR rate↑ |
| Temperature | 5 | Peripheral changes |
| Respiration | 5 | Rate↑, variability↓ |
| Accelerometer | 5 | Movement artifacts |

### Next Steps

- Week 4: Feature selection and dimensionality reduction
- Week 5: ML model training (Random Forest, SVM, XGBoost)
- Week 6: Deep learning models (CNN with image encoding)

---

*CalmSense: Multimodal Stress Detection using Physiological Signals*